# Sweeping an input value

A parameter sweep changes a constant the flow reads with `param()`; an input sweep changes a record's own
value. `overrides` without `at` writes an input column before anything runs; overriding a value a step
produces needs `at=<step path>`.

**How an experiment runs.** The runner feeds the *baseline* (the unchanged flow) and every *scenario* the
same records, one batch at a time. Each **metric** gets its own copy per scenario and collects through hooks:
a *global* metric reads the flow's output; a *step* metric (`nodes="<glob>"`) reads what matching steps
wrote, on the records that ran them. `summarise()` gives a typed summary; a **plot** draws one metric's
summaries across scenarios. `result.show()` lays it all out.

In [1]:
from decikit.experiments import *
from example_projects.experiments.loan_pipeline import applications, build

ENTRY = "example_projects.experiments.loan_pipeline:build"
pipeline = build()   # ratio -> rate -> pricing (secured | unsecured) -> approved, payment
loans = applications(200_000)
loans.head()

income,debt,collateral
f64,f64,f64
3829.0,718.0,1344.0
3454.0,938.0,3913.0
4704.0,1956.0,0.0
3797.0,1960.0,16821.0
2939.0,387.0,0.0


In [2]:
exp = (Experiment("collateral", pipeline, loans, entry=ENTRY)
       .scenario("all_secured", overrides={"collateral": 5_000.0})
       .scenario("none_secured", overrides={"collateral": 0.0})
       .scenario("cheap_base_rate", at="loans/rate", overrides={"rate": 0.05})
       .metric(Divergence(column="approved"),
               Histogram(name="rate", column="rate", nodes="loans/pricing", range=(0.0, 0.3)),
               BranchFlows(outcome="approved"))
       .plot(Changes(metric="divergence"), Distribution(metric="rate"),
             Sankey(name="flows_none_secured", metric="branch_flows", scenario="none_secured")))
result = exp.run()
result.plots["rate"].show()

<vega-lite chart>

`validate` returns every problem with a definition at once, before anything runs.

In [3]:
broken = (Experiment("broken", pipeline, loans)
          .scenario("no_point", overrides={"rate": 0.1})
          .metric(ValueCounts(column="approved"))
          .plot(Changes(metric="value_counts")))
validate(broken.def_, pipeline)

["no_point: override 'rate' is produced, not an input; declare 'at: <step path>'",
 "plot 'value_counts' needs a Divergences summary; metric 'value_counts' gives Counts"]

In [4]:
result.show()